# Episode 13: Deep Networks and Backpropagation
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2013%20-%20Backpropagation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 13, one step per section. The first eight steps use only `numpy` and the standard library (already installed in Colab), so choose **Runtime, Run all**. The last section downloads GPT-2 small's published weights (548 MB, once) to confirm the token IDs and the parameter counts; skip it on a slow connection.

Covers: the forward pass, the loss, the backward pass (blame, output first, with the calculus chain rule), the check by nudging, one gradient-descent step, the cost of the backward pass, the depth preview (ten sigmoid layers), the word loss p − q, and the lookup that hands blame to the embedding table.

*Blame* on a number x is ∂ℓ/∂x, one entry of the gradient: a sensitivity, not a moral judgment. Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart. The network is Episode 12's hand-set 2 × 2 example, unchanged; its weights are illustrative, not learned.

In [1]:
import math
import numpy as np

def sigma(z):                          # the sigmoid, sigma(z) = 1 / (1 + exp(-z))
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):                           # the bend psi = ReLU, entry by entry
    return np.maximum(z, 0.0)

def relu_slope(z):                     # psi'(z): 0 for z <= 0, 1 for z > 0 (no pre-activation here is exactly 0)
    return (z > 0).astype(float)

# Episode 12's network, unchanged: layer 1 is u -> h -> psi(h); layer 2 is psi(h) -> h'
P0 = dict(W=np.array([[1.0, -1.0], [2.0, 1.0]]), beta=np.array([-1.0, 0.0]),
          Wp=np.array([[1.0, 2.0], [1.0, -1.0]]), betap=np.array([2.0, 0.0]))
u0 = np.array([1.0, 2.0])
y0 = np.array([9.0, -2.0])             # the target of the squared-error loss

def forward(P, u):
    Wu = P["W"] @ u
    h = Wu + P["beta"]                 # pre-activation (Episode 3's linear-layer output)
    a = relu(h)                        # a = psi(h), the activated output
    Wpa = P["Wp"] @ a
    hp = Wpa + P["betap"]              # h', the second layer's output
    return dict(Wu=Wu, h=h, a=a, Wpa=Wpa, hp=hp)

def loss(P, u, y):                     # squared error for one term
    hp = forward(P, u)["hp"]
    return float(np.sum((hp - y) ** 2))
print("network ready")

network ready


## Step 1: forward
Wu, h = Wu + β, ψ(h), W′ψ(h), h′.

In [2]:
c = forward(P0, u0)
print("Wu =", c["Wu"].tolist(), " h =", c["h"].tolist(), " psi(h) =", c["a"].tolist(), " W'psi(h) =", c["Wpa"].tolist(), " h' =", c["hp"].tolist())
assert c["Wu"].tolist() == [-1, 4] and c["h"].tolist() == [-2, 4]
assert c["a"].tolist() == [0, 4] and c["Wpa"].tolist() == [8, -4] and c["hp"].tolist() == [10, -4]

Wu = [-1.0, 4.0]  h = [-2.0, 4.0]  psi(h) = [0.0, 4.0]  W'psi(h) = [8.0, -4.0]  h' = [10.0, -4.0]


## Step 2: loss and target
Squared error for one term, ℓ = (h′₁ − y₁)² + (h′₂ − y₂)², with y = (9, −2). A softmax on (10, −4) would be degenerate, so the Lab uses squared error.

In [3]:
ell = loss(P0, u0, y0)
print("loss l =", ell)
assert ell == 5.0 and (10 - 9) ** 2 + (-4 + 2) ** 2 == 5

hp = c["hp"]
p_first = math.exp(hp[0]) / (math.exp(hp[0]) + math.exp(hp[1]))
print("softmax of (10, -4), first score:", round(p_first, 7))
assert round(p_first, 7) == 0.9999992

loss l = 5.0
softmax of (10, -4), first score: 0.9999992


## Step 3: backward, output first
At every row, blame on a number = (blame on what it feeds) × (how fast that changes per unit change of the number). That is the calculus chain rule. If a number feeds two things, the shares add.

In [4]:
def backward(P, u, y):
    c = forward(P, u)
    g = {}
    g["hp"] = 2 * (c["hp"] - y)                    # blame on h'
    g["betap"] = g["hp"].copy()                    # dh'/dbeta' = 1
    g["Wp"] = np.outer(g["hp"], c["a"])            # dh'_i/dW'_ij = psi(h)_j
    g["a"] = P["Wp"].T @ g["hp"]                   # dh'_i/dpsi(h)_j = W'_ij, summed over i (the sum over paths)
    g["h"] = relu_slope(c["h"]) * g["a"]           # psi'(h) times the blame on psi(h)
    g["beta"] = g["h"].copy()                      # dh_i/dbeta_i = 1
    g["W"] = np.outer(g["h"], u)                   # dh_i/dW_ij = u_j
    g["u"] = P["W"].T @ g["h"]                     # dh_i/du_j = W_ij, summed over i
    return g

g = backward(P0, u0, y0)
for name in ("hp", "betap", "Wp", "a", "h", "beta", "W", "u"):
    print(f"{name:6s}", g[name].tolist())
assert g["hp"].tolist() == [2, -4] and g["betap"].tolist() == [2, -4]
assert g["Wp"].tolist() == [[0, 8], [0, -16]]
assert g["a"].tolist() == [-2, 8]
assert g["h"].tolist() == [0, 8] and g["beta"].tolist() == [0, 8]
assert g["W"].tolist() == [[0, 0], [8, 16]]
assert g["u"].tolist() == [16, 8]

# the sum over paths: psi(h)_2 feeds both outputs
along_1 = P0["Wp"][0, 1] * g["hp"][0]              # 2 * 2
along_2 = P0["Wp"][1, 1] * g["hp"][1]              # (-1) * (-4)
print("two paths into psi(h)_2:", along_1, "+", along_2, "=", along_1 + along_2)
assert (along_1, along_2, along_1 + along_2) == (4, 4, 8) == (4, 4, g["a"][1])

# neuron 1 is off (h_1 = -2), so its slope is 0: nothing flows back through it
assert relu_slope(c["h"]).tolist() == [0, 1] and g["W"][0].tolist() == [0, 0] and g["beta"][0] == 0

hp     [2.0, -4.0]
betap  [2.0, -4.0]
Wp     [[0.0, 8.0], [-0.0, -16.0]]
a      [-2.0, 8.0]
h      [-0.0, 8.0]
beta   [-0.0, 8.0]
W      [[-0.0, -0.0], [8.0, 16.0]]
u      [16.0, 8.0]
two paths into psi(h)_2: 4.0 + 4.0 = 8.0


## Step 4: check by nudging
Nudge each of the 14 numbers (12 parameters and u) by ±10⁻⁵, rerun the forward pass, and divide the change in ℓ by 2 × 10⁻⁵ (a central difference).

In [5]:
def pack(P, u):
    return np.concatenate([P["W"].ravel(), P["beta"], P["Wp"].ravel(), P["betap"], u])

def unpack(v):
    P = dict(W=v[0:4].reshape(2, 2), beta=v[4:6], Wp=v[6:10].reshape(2, 2), betap=v[10:12])
    return P, v[12:14]

def loss_vec(v):
    P, u = unpack(v)
    return loss(P, u, y0)

v0 = pack(P0, u0)
assert v0.size == 14
eps = 1e-5
numeric = np.array([(loss_vec(v0 + eps * e) - loss_vec(v0 - eps * e)) / (2 * eps) for e in np.eye(14)])
analytic = pack({k: g[k] for k in ("W", "beta", "Wp", "betap")}, g["u"])
err = np.abs(numeric - analytic)
print("largest difference over 14 numbers:", err.max())
assert err.max() < 1e-6

largest difference over 14 numbers: 2.1404478189879228e-10


## Step 5: one step
θ ← θ − η ∇_θ ℓ on the 12 parameters (u is data). η = 0.01 lowers ℓ; the same direction with η = 0.05 overshoots: blame says which way, η says how far.

In [6]:
def step(P, grads, eta):
    return {k: P[k] - eta * grads[k] for k in ("W", "beta", "Wp", "betap")}

P1 = step(P0, g, 0.01)
print("W' =", P1["Wp"].tolist(), " beta' =", P1["betap"].tolist(), " W =", P1["W"].tolist(), " beta =", P1["beta"].tolist())
assert np.allclose(P1["Wp"], [[1, 1.92], [1, -0.84]]) and np.allclose(P1["betap"], [1.98, 0.04])
assert np.allclose(P1["W"], [[1, -1], [1.92, 0.84]]) and np.allclose(P1["beta"], [-1, -0.08])

c1 = forward(P1, u0)
print("Wu =", c1["Wu"].tolist(), " h =", c1["h"].tolist(), " psi(h) =", c1["a"].tolist(), " h' =", c1["hp"].tolist())
assert np.allclose(c1["Wu"], [-1, 3.6]) and np.allclose(c1["h"], [-2, 3.52]) and np.allclose(c1["a"], [0, 3.52])
assert np.allclose(c1["hp"], [8.7384, -2.9168])

ell_1 = loss(P1, u0, y0)
print("loss after one step, eta = 0.01:", round(ell_1, 4))
assert round(ell_1, 3) == 0.909 and round((8.7384 - 9) ** 2, 4) == 0.0684 and round((-2.9168 + 2) ** 2, 4) == 0.8405

ell_5 = loss(step(P0, g, 0.05), u0, y0)
print("loss after one step, eta = 0.05:", round(ell_5, 2), "(up from 5)")
assert round(ell_5, 1) == 24.1 and ell_5 > ell

W' = [[1.0, 1.92], [1.0, -0.84]]  beta' = [1.98, 0.04]  W = [[1.0, -1.0], [1.92, 0.84]]  beta = [-1.0, -0.08]
Wu = [-1.0, 3.5999999999999996]  h = [-2.0, 3.5199999999999996]  psi(h) = [0.0, 3.5199999999999996]  h' = [8.738399999999999, -2.9167999999999994]
loss after one step, eta = 0.01: 0.909
loss after one step, eta = 0.05: 24.15 (up from 5)


## Step 6: cost
Count the scalar multiplications in the matrix products (not FLOPs, and not a general ratio). The counter multiplies rows × columns × inner size for each product.

In [7]:
def mm_cost(shape_a, shape_b):         # scalar multiplications in a (r x k) times (k x c) product
    (r, k), (k2, cc) = shape_a, shape_b
    assert k == k2
    return r * k * cc

# forward: W u (2x2 times 2x1) and W' psi(h) (2x2 times 2x1)
forward_cost = mm_cost((2, 2), (2, 1)) + mm_cost((2, 2), (2, 1))
# backward: blame on W' (outer product 2x1 times 1x2), on psi(h) (W'^T times 2x1), on W (outer product), on u (W^T times 2x1)
bw_Wp = mm_cost((2, 1), (1, 2)); bw_a = mm_cost((2, 2), (2, 1)); bw_W = mm_cost((2, 1), (1, 2)); bw_u = mm_cost((2, 2), (2, 1))
backward_cost = bw_Wp + bw_a + bw_W + bw_u
no_u_row = backward_cost - bw_u        # Step 5 does not use the blame on u (u is data)
print("forward", forward_cost, " backward", backward_cost, " backward without the u row", no_u_row)
assert forward_cost == 8 and backward_cost == 16 and no_u_row == 12
assert backward_cost == 2 * forward_cost and no_u_row / forward_cost == 1.5 and forward_cost + backward_cost == 3 * forward_cost

forward 8  backward 16  backward without the u row 12


## Step 7: depth (preview)
Ten layers in a row, each computing σ(w·a + b) from the layer below's output a, starting from a = 1/2, with b = −w/2 so every pre-activation is 0. Each layer passes blame back scaled by w times the sigmoid's slope at 0, which is w/4. The backward pass multiplies those ten factors. The toy shares w in every layer, the case Goodfellow warns about.

In [8]:
def depth_forward(a0, w, layers=10):
    a, outs, pres = a0, [], []
    for _ in range(layers):
        z = w * a - w / 2                  # pre-activation
        pres.append(z)
        a = float(sigma(z))
        outs.append(a)
    return a, outs, pres

def depth_backward(outs, w):
    blame = 1.0                            # blame on the final output
    for s in reversed(outs):               # one factor per layer: w times the sigmoid's slope s(1 - s)
        blame *= w * s * (1 - s)
    return blame

for w, expected, h in ((1.0, 0.25 ** 10, 1e-4), (8.0, 1024.0, 1e-6)):     # w = 8 curves sharply, so it needs a smaller nudge h
    _, outs, pres = depth_forward(0.5, w)
    assert all(z == 0.0 for z in pres)     # every pre-activation is exactly 0
    bp = depth_backward(outs, w)
    fd = (depth_forward(0.5 + h, w)[0] - depth_forward(0.5 - h, w)[0]) / (2 * h)
    print(f"w = {w}: backward product = {bp:.6g}, (w/4)^10 = {(w / 4) ** 10:.6g}, nudge check = {fd:.6g}")
    assert abs(bp - expected) / expected < 1e-12 and abs(fd - expected) / expected < 1e-4
assert f"{0.25 ** 10:.2e}" == "9.54e-07" and (8 / 4) ** 10 == 1024

# the other order of the same toy, h_{k+1} = w * sigma(h_k) + b with h_0 = 0, has the same factor
for w in (1.0, 8.0):
    hk, factor = 0.0, 1.0
    for _ in range(10):
        s = float(sigma(hk))
        factor *= w * s * (1 - s)
        hk = w * s - w / 2
        assert hk == 0.0
    assert abs(factor - (w / 4) ** 10) / (w / 4) ** 10 < 1e-12

# with random pre-activations and w = 1 the product never exceeds (1/4)^10 (the sigmoid's slope is at most 1/4)
rng = np.random.default_rng(3)
for _ in range(1000):
    xs = rng.normal(size=10) * 3
    assert np.prod(sigma(xs) * (1 - sigma(xs))) <= 0.25 ** 10 * (1 + 1e-12)
print("depth checks passed")

w = 1.0: backward product = 9.53674e-07, (w/4)^10 = 9.53674e-07, nudge check = 9.53675e-07
w = 8.0: backward product = 1024, (w/4)^10 = 1024, nudge check = 1024
depth checks passed


## Section 3: the " tired" beat
Give " tired" probability 0.10 (Episode 10's illustrative number; surprisal −ln 0.10 = 2.303 nats). A word loss ℓ = −ln pₖ starts its backward pass at p − q (q one-hot), not at 2(h′ − y). The scores are GPT-2 small's 50,257 numbers; the numbers here are illustrative.

In [9]:
V = 50_257                             # |V|
tired_id = 10032
rng = np.random.default_rng(0)
p = rng.random(V); p[tired_id] = 0.0
p = p / p.sum() * 0.90                 # the other tokens share the remaining 0.90
p[tired_id] = 0.10
assert abs(p.sum() - 1) < 1e-12
scores = np.log(p)                     # any scores whose softmax is p
q = np.zeros(V); q[tired_id] = 1.0

print("surprisal -ln 0.10 =", round(-math.log(0.10), 3))
assert round(-math.log(0.10), 3) == 2.303

def word_loss(s):                      # l = -ln softmax(s)[tired] = -s_k + ln sum exp(s_j)
    m = s.max()
    return float(-(s[tired_id]) + m + np.log(np.exp(s - m).sum()))

blame_scores = p - q
print("blame on the score of ' tired':", round(float(blame_scores[tired_id]), 2), " sum of all", V, "entries:", float(blame_scores.sum()))
assert round(float(blame_scores[tired_id]), 2) == -0.90 and abs(blame_scores.sum()) < 1e-12 and blame_scores.size == 50_257
assert np.allclose(np.delete(blame_scores, tired_id), np.delete(p, tired_id))

# nudge six of the scores (the target, and five others) and compare with p - q
e = 1e-6
for j in [tired_id, 0, 1, 2, 3, 40_000]:
    s_plus, s_minus = scores.copy(), scores.copy()
    s_plus[j] += e; s_minus[j] -= e
    fd = (word_loss(s_plus) - word_loss(s_minus)) / (2 * e)
    assert abs(fd - blame_scores[j]) < 1e-8
print("p - q agrees with nudging on 6 scores")

# random score vectors: p - q matches nudging on every entry (small vocabulary)
rs = np.random.default_rng(1)
for _ in range(20):
    s = rs.normal(size=7) * 2; k = int(rs.integers(7))
    pp = np.exp(s - s.max()); pp /= pp.sum(); qq = np.zeros(7); qq[k] = 1
    def L(sv, k=k):
        return float(-(sv[k]) + sv.max() + np.log(np.exp(sv - sv.max()).sum()))
    fd = np.array([(L(s + e * np.eye(7)[j]) - L(s - e * np.eye(7)[j])) / (2 * e) for j in range(7)])
    assert np.allclose(fd, pp - qq, atol=1e-7) and abs((pp - qq).sum()) < 1e-12
print("20 random score vectors passed")

surprisal -ln 0.10 = 2.303
blame on the score of ' tired': -0.9  sum of all 50257 entries: -1.1102230246251565e-16
p - q agrees with nudging on 6 scores
20 random score vectors passed


### The lookup hands blame to the embedding table
U = O W_E picks rows of the embedding table W_E. Only the rows of tokens that were read get blame, and a token read twice gets two shares added. Toy: |V| = 5 tokens, d = 2, token IDs (3, 0, 3).

In [10]:
rl = np.random.default_rng(7)
Vt, d = 5, 2
WE = rl.normal(size=(Vt, d))
ids = [3, 0, 3]
O = np.zeros((len(ids), Vt))
for r, i in enumerate(ids):
    O[r, i] = 1.0                      # one-hot rows: U = O W_E
wgt = rl.normal(size=(len(ids), d))    # any downstream use of U: loss = sum(wgt * U)

def toy_loss(WE_):
    return float(np.sum(wgt * (O @ WE_)))

G_U = wgt                              # blame on U
G_WE = O.T @ G_U                       # blame on W_E: O-transpose times the blame on U
print("blame on W_E rows:", [bool(np.any(G_WE[i])) for i in range(Vt)])
assert [bool(np.any(G_WE[i])) for i in range(Vt)] == [True, False, False, True, False]   # only rows 0 and 3
assert np.allclose(G_WE[3], G_U[0] + G_U[2]) and np.allclose(G_WE[0], G_U[1])             # row 3 is the sum of two shares

fd = np.zeros_like(WE)
for i in range(Vt):
    for j in range(d):
        E = np.zeros_like(WE); E[i, j] = 1e-6
        fd[i, j] = (toy_loss(WE + E) - toy_loss(WE - E)) / 2e-6
assert np.allclose(fd, G_WE, atol=1e-8)

# the " tired" term: nine context tokens, nine different IDs, so exactly 9 rows of W_E are read
GPT2_IDS = [464, 3797, 3332, 319, 262, 2603, 780, 340, 373, 10032, 13]    # The cat sat on the mat because it was tired .
assert len(GPT2_IDS) == 11 and len(set(GPT2_IDS)) == 11                   # n = 11 tokens, N = 10 terms
context = GPT2_IDS[:9]
assert GPT2_IDS[9] == 10032 and len(set(context)) == 9
print("rows of W_E read by the ' tired' term:", len(set(context)))

blame on W_E rows: [True, False, False, True, False]
rows of W_E read by the ' tired' term: 9


## Check question
Same network, target y = (10, −2). (a) The blame on h′, on β′ and on W′. (b) Why row 1 of W got blame 0 in the Lab. (c) How many blame numbers one gradient of GPT-2 small holds.

In [11]:
y_alt = np.array([10.0, -2.0])
ga = backward(P0, u0, y_alt)
print("(a) blame on h' and beta':", ga["hp"].tolist(), ga["betap"].tolist(), " on W':", ga["Wp"].tolist())
assert ga["hp"].tolist() == [0, -4] and ga["betap"].tolist() == [0, -4] and ga["Wp"].tolist() == [[0, 0], [0, -16]]
assert c["h"][0] == -2 and relu_slope(c["h"])[0] == 0 and g["W"][0].tolist() == [0, 0]   # (b) h_1 = -2, so ReLU's slope is 0
total_params = 124_439_808
print("(c) one blame number per parameter:", total_params)
print("ALL MATH LAB CHECKS PASSED")

(a) blame on h' and beta': [0.0, -4.0] [0.0, -4.0]  on W': [[0.0, 0.0], [-0.0, -16.0]]
(c) one blame number per parameter: 124439808
ALL MATH LAB CHECKS PASSED


## Real scale: GPT-2 small
This cell confirms three things against the published weights (Hugging Face `openai-community/gpt2`, 548 MB, once): the 11 token IDs of the running sentence, the parameter count (124,439,808, which is the number of blame numbers in one gradient), and the shape of the embedding table. The checkpoint file holds no separate output matrix (`lm_head`): the output layer reuses the embedding table, as nanoGPT's code states (this checkpoint was not compared with OpenAI's original release). Skip this section on a slow connection.

In [12]:
import subprocess, sys
for pkg in ["tiktoken", "safetensors", "huggingface_hub"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import tiktoken
from huggingface_hub import hf_hub_download
from safetensors import safe_open

enc = tiktoken.get_encoding("gpt2")
toks = enc.encode("The cat sat on the mat because it was tired.")
print(len(toks), "tokens:", toks)
assert toks == GPT2_IDS and enc.decode([toks[9]]) == " tired"

path = hf_hub_download("openai-community/gpt2", "model.safetensors", local_dir="gpt2_weights")
f = safe_open(path, framework="np")
shape = lambda name: tuple(f.get_slice(name).get_shape())
size = lambda name: int(np.prod(shape(name)))
keys = list(f.keys())
total = sum(size(k) for k in keys if not k.endswith(".attn.bias"))      # the 12 attention-mask buffers are not parameters
print("parameters:", total, " embedding table:", shape("wte.weight"))
assert total == total_params == 124_439_808
assert shape("wte.weight") == (50_257, 768)
assert not any("lm_head" in k for k in keys)                           # no separate output matrix in the file
assert len({k.split(".")[1] for k in keys if k.startswith("h.")}) == 12   # 12 blocks
print("ALL EPISODE 12 CHECKS PASSED")

11 tokens: [464, 3797, 3332, 319, 262, 2603, 780, 340, 373, 10032, 13]


parameters: 124439808  embedding table: (50257, 768)
ALL EPISODE 12 CHECKS PASSED
